# Data Collection

This notebook pulls together two independent data sources needed for the recommender:

- **Dataset A**: anime/manga content metadata (titles, genres, synopses) from the AniList GraphQL API
- **Dataset B**: real user-anime rating interactions from Kaggle, which power collaborative filtering

It also builds the implicit-feedback label (`is_positive`) and the train/test split used by every downstream notebook.

In [1]:
import psutil, platform
print(platform.node())
print(psutil.virtual_memory().total / 1e9, "GB")

Armands-MacBook-Pro.local
8.589934592 GB


In [4]:
from sklearn.model_selection import train_test_split
import time
import json
import os
import requests
import kagglehub
import numpy as np
import pandas as pd
import re

DATA_DIR = os.path.join("..", "data")  
ANIME_DATA_PATH = os.path.join(DATA_DIR, "anime_data.jsonl")
MANGA_DATA_PATH = os.path.join(DATA_DIR, "manga_data.jsonl")
MAL_ID_MAP_PATH = os.path.join(DATA_DIR, "anilist_to_mal.json")

ANILIST_URL = "https://graphql.anilist.co" #Getting data from AniList API

# Download latest version
path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

## Dataset A: Anime & Manga Metadata (AniList)

Pulls the top 5,000 anime and top 5,000 manga by popularity from AniList's public GraphQL API (no auth required).

**Note:** AniList caps pagination depth at 5,000 entries (`perPage: 50` × 100 pages) — this is a hard API limit, not a bug, so the loop intentionally stops at page 100. Results are written incrementally to `.jsonl` (checkpointed every 10 pages) so a crash mid-run doesn't lose already-fetched data. `time.sleep(2)` between requests avoids the API's rate limit (429 errors).

Output: `data/anime_data.jsonl`, `data/manga_data.jsonl`

In [5]:
if os.path.exists(ANIME_DATA_PATH):
    with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
        anime_content = [json.loads(line) for line in f]
else:
    anime_content = []

if os.path.exists(MAL_ID_MAP_PATH):
    with open(MAL_ID_MAP_PATH, "r") as f:
        mal_ids = {int(k): v for k, v in json.load(f).items()}
else:
    mal_ids = {}

In [ ]:
#Datasets A - Anime and Manga: IDs, description, genres, etc

    
    
QUERY = """
query ($page: Int, $type: MediaType) {
  Page(page: $page, perPage: 50) {
    pageInfo { hasNextPage }
    media(type: $type, sort: POPULARITY_DESC) {
      id
      title { romaji english }
      genres
      tags { name }
      description
      coverImage { large }
      averageScore
      popularity
      format
    }
  }
}
"""


def fetch_page(page, media_type="ANIME"):
    variables = {"page": page, "type": media_type}
    response = requests.post(ANILIST_URL, json={"query": QUERY, "variables": variables})
    response.raise_for_status()
    return response.json()["data"]["Page"]
i=1
data = []
#Get Anime data
while True:
    x=fetch_page(i)
    data.extend(x['media'])
    
    if i == 100: #API caps at 5000 requests 
        with open(ANIME_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        break
    i+=1
    
    if i % 10 == 0:
        with open(ANIME_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        data.clear()
        
    time.sleep(2) #Ensures we wont have too many requests too quick

data.clear()
i=1
#Get Manga data
while True:
    x=fetch_page(i, media_type="MANGA")
    data.extend(x['media'])
    
    if i == 100: 
        with open(MANGA_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        break
    i+=1
    
    if i % 10 == 0:
        with open(MANGA_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        data.clear()
        
    time.sleep(2) 



In [6]:
MAL_ID_BATCH_QUERY = """
query ($ids: [Int]) {
  Page(page: 1, perPage: 50) {
    media(id_in: $ids, type: ANIME) {
      id
      idMal
    }
  }
}
"""

def fetch_mal_ids_batch(anilist_ids, max_retries=5):
    variables = {"ids": anilist_ids}
    wait = 5
    for attempt in range(max_retries):
        response = requests.post(ANILIST_URL, json={"query": MAL_ID_BATCH_QUERY, "variables": variables})
        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", wait))
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            wait *= 2
            continue
        response.raise_for_status()
        return response.json()["data"]["Page"]["media"]
    raise RuntimeError(f"Still rate limited after {max_retries} retries")

# Resume from a previous (possibly interrupted) run instead of re-fetching everything


remaining_ids = [a['id'] for a in anime_content if a['id'] not in mal_ids]
batch_size = 50

for start in range(0, len(remaining_ids), batch_size):
    batch = remaining_ids[start:start + batch_size]
    try:
        media_list = fetch_mal_ids_batch(batch)
        found = {m['id']: m['idMal'] for m in media_list}
        for aid in batch:
            mal_ids[aid] = found.get(aid)
    except Exception as e:
        for aid in batch:
            mal_ids[aid] = None
        print(f"Failed on batch starting {start}: {e}")

    print(f"{start + len(batch)}/{len(remaining_ids)}")
    with open(MAL_ID_MAP_PATH, "w") as f:
        json.dump(mal_ids, f)

    time.sleep(2)  # AniList allows ~90 req/min; 50-id batches + 2s spacing keeps us well under that

In [7]:
with open(os.path.join(DATA_DIR, "anilist_to_mal.json"), "r") as f:
    mal_ids = json.load(f)

print(len(mal_ids))
none_count = sum(1 for v in mal_ids.values() if v is None)
print(f"{none_count} anime failed to get a MAL ID")

animes['mal_id'] = animes['mal_url'].str.extract(r'/anime/(\d+)').astype(int)

anilist_mal_ids = set(v for v in mal_ids.values() if v is not None)
ratings_mal_ids = set(animes['mal_id'])

overlap = anilist_mal_ids & ratings_mal_ids
print(f"AniList MAL IDs: {len(anilist_mal_ids)}")
print(f"Ratings dataset MAL IDs: {len(ratings_mal_ids)}")
print(f"Overlap: {len(overlap)}")

4950
11 anime failed to get a MAL ID
AniList MAL IDs: 4936
Ratings dataset MAL IDs: 20237
Overlap: 4736


## Dataset B: User Ratings (Kaggle)

Downloads `ramazanturann/user-animelist-dataset` via `kagglehub`, which contains 148M user-anime rating pairs across ~1.77M users and 20,237 anime, sourced from MAL/AniList/Kitsu combined.

Loads three things:
- `animes.csv` — this dataset's own anime metadata (used for displaying human-readable titles later)
- `genres` — a slim `[animeID, genres]` subset of the above
- `ratings.npy` → `ratings_df` — the raw interaction data, loaded from the `.npy` binary format (much faster/smaller than the equivalent `.csv`, at the cost of needing to manually assign column names: `[user_id, anime_id, rating]`)

**Confirmed:** `animes['animeID']` and `ratings_df['anime_id']` use the same ID system (verified 100% overlap — 20,237/20,237) so they can be joined directly.

In [8]:
print(animes.shape)
display(animes.head())

# Genres
genres = animes[['animeID', 'genres']].copy() 
display(genres)

# Ratings
ratings_array = np.load(os.path.join(path, "ratings.npy"))
ratings_df = pd.DataFrame(ratings_array, columns=["user_id", "anime_id", "rating"])

display(ratings_df[:25])



(20237, 13)


,animeID,title,alternative_title,type,year,score,episodes,mal_url,sequel,image_url,genres,genres_detailed,mal_id
0,1,Howl's Moving Castle,Howl no Ugoku Shiro,MOVIE,2004,8.41,1,https://myanimelist.net/anime/431,False,https://cdn.myanimelist.net/images/anime/1470/...,"['Adventure', 'Award Winning', 'Drama', 'Fanta...","['action', 'adventure', 'age gap', 'air force'...",431
1,2,Death Note,NaN,TV,2006,8.63,37,https://myanimelist.net/anime/1535,False,https://cdn.myanimelist.net/images/anime/1079/...,"['Supernatural', 'Suspense']","['achronological order', 'acting', 'adapted in...",1535
2,3,Problem Children Are Coming from Another World...,Mondaiji-tachi ga Isekai kara Kuru Sou desu yo?,TV,2013,7.42,10,https://myanimelist.net/anime/15315,False,https://cdn.myanimelist.net/images/anime/12/43...,"['Action', 'Comedy', 'Fantasy']","['action', 'alternative world', 'anthropomorph...",15315
3,4,BTOOOM!,Btooom!,TV,2012,7.34,12,https://myanimelist.net/anime/14345,False,https://cdn.myanimelist.net/images/anime/4/409...,"['Action', 'Sci-Fi', 'Suspense']","['achronological order', 'action', 'adventure'...",14345
4,5,Sword Art Online,NaN,TV,2012,7.5,25,https://myanimelist.net/anime/11757,False,https://cdn.myanimelist.net/images/anime/11/39...,"['Action', 'Adventure', 'Fantasy', 'Romance']","['action', 'action drama', 'adventure', 'alter...",11757


,animeID,genres
0,1,"['Adventure', 'Award Winning', 'Drama', 'Fanta..."
1,2,"['Supernatural', 'Suspense']"
2,3,"['Action', 'Comedy', 'Fantasy']"
3,4,"['Action', 'Sci-Fi', 'Suspense']"
4,5,"['Action', 'Adventure', 'Fantasy', 'Romance']"
...,...,...
20232,20233,['Fantasy']
20233,20234,['Fantasy']
20234,20235,['Fantasy']
20235,20236,[]


,user_id,anime_id,rating
0,1,1,10
1,1,2,10
2,1,3,7
3,1,4,10
4,1,5,10
5,1,6,10
6,1,7,10
7,1,8,10
8,1,9,6
9,1,10,10


## Breaking down the numbers from Dataset B
- Rating distrubuition
- Ratings per user
- Ratings per anime

In [6]:
print(f"Rating Distrubition: {ratings_df['rating'].value_counts().sort_index()}\n")
print(f"Ratings per user: {ratings_df.groupby('user_id').size().describe()}\n")
print(f"Ratings per anime: {ratings_df.groupby('anime_id').size().describe()}\n")

Rating Distrubition: rating
0       186337
1      1643276
2      1273086
3      2008890
4      4750391
5      7471134
6     16391767
7     30726885
8     35294326
9     21767496
10    26656908
Name: count, dtype: int64

Ratings per user: count    1.774522e+06
mean     8.349882e+01
std      1.560422e+02
min      5.000000e+00
25%      1.000000e+01
50%      2.900000e+01
75%      9.200000e+01
max      1.088100e+04
dtype: float64

Ratings per anime: count     20237.000000
mean       7321.761921
std       32883.434898
min           1.000000
25%          49.000000
50%         264.000000
75%        2127.000000
max      956713.000000
dtype: float64



## Converting Ratings to Implicit Feedback

Real recommender systems (Netflix, Spotify, etc.) mostly work from *implicit* signals (watched/clicked), not clean explicit ratings — so we convert this dataset's 1-10 scores into a binary `is_positive` label to match that framing, and because the downstream model (`implicit` library's ALS) is built specifically for implicit feedback.

- Ratings of `0` are dropped — likely a placeholder/"unrated" marker rather than a genuine score of zero (unconfirmed, but 0 doesn't fit the 1-10 scale the rest of the data uses)
- **Threshold set at 8+ = positive**, not 7+. MAL ratings skew heavily positive (~77% of all ratings are 7+, per the distribution above) — a 7+ cutoff would label the vast majority of interactions "positive" and give the model little contrast to learn from. 8+ yields a healthier ~57/43 split.

In [9]:
ratings_clean = ratings_df[ratings_df['rating'] != 0].copy()
ratings_clean['is_positive'] = (ratings_clean['rating'] >= 8).astype(int) #Cut off at 8 due to how people rate shows. People tend to rate higher so to counteract this shift cut off will be at 8

print(ratings_clean['is_positive'].value_counts(normalize=True))

is_positive
1    0.565728
0    0.434272
Name: proportion, dtype: float64
(147984159, 4)
4.143556452 GB
svmem(total=8589934592, available=1260699648, percent=85.3, used=3070754816, free=68141056, active=1202372608, inactive=1111506944, wired=1868382208)


In [ ]:
ratings_clean['user_id'] = ratings_clean['user_id'].astype('uint32')
ratings_clean['anime_id'] = ratings_clean['anime_id'].astype('uint16')
ratings_clean['rating'] = ratings_clean['rating'].astype('int8')
ratings_clean['is_positive'] = ratings_clean['is_positive'].astype('int8')

idx = np.arange(len(ratings_clean))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)
train = ratings_clean.iloc[train_idx].reset_index(drop=True)
test = ratings_clean.iloc[test_idx].reset_index(drop=True)
del ratings_clean  

print(train.memory_usage(deep=True).sum() / 1e9, "GB")
print(test.memory_usage(deep=True).sum() / 1e9, "GB")

## Train/Test Split

**Important limitation:** this dataset does not include interaction timestamps (confirmed in the source dataset's own documentation — "MovieLens format except timestamp"). A proper recommender evaluation should use a *time-based* split (train on the past, test on predicting the future) to avoid leaking future behavior into training. Without timestamps, that's not possible here.

**Fallback used instead: a random 80/20 split.** 

Saved as `.parquet` (not `.csv`) for fast, efficient reloading in later notebooks — both files are gitignored, since they're fully reproducible by rerunning this cell.

In [ ]:
train, test = train_test_split(ratings_clean, test_size=0.2, random_state=42)

train.to_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test.to_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

## Manga Ratings (Cross-Domain Data)

A second, separate ratings dataset (`ddmasterdon/manga-2023-details-with-review`) — real user-manga scores, ~1.2M rows across 29,806 users and 22,346 manga.

**Known limitation:** this dataset's `user` field is a MAL *username*, not a numeric ID, and there is no reliable way to confirm a user here is the same person as a given `user_id` in the anime ratings dataset above. **This means true identity-linked cross-domain personalization ("recommend manga based on this exact person's anime behavior") is not possible with these two datasets.** The cross-domain bridge in this project is instead built on *content similarity* (shared genres/embeddings between an anime a user liked and a manga with a similar profile), not shared user identity — a deliberate, documented design choice, not an oversight.

This manga ratings data is used to build a *separate*, smaller-scale ALS model for manga specifically (not linked to the anime model).

In [9]:
manga_path = kagglehub.dataset_download("ddmasterdon/manga-2023-details-with-review")
manga_ratings = pd.read_csv(os.path.join(manga_path, "manga_ratinga.csv"))  # adjust filename if different

manga_ratings.head()

manga_ratings['manga_idx'] = manga_ratings['manga_id'].astype('category').cat.codes
manga_ratings['user_idx'] = manga_ratings['user'].astype('category').cat.codes

manga_id_map = dict(enumerate(manga_ratings['manga_id'].astype('category').cat.categories))
manga_user_map = dict(enumerate(manga_ratings['user'].astype('category').cat.categories))

manga_ratings.to_parquet(os.path.join(DATA_DIR, "manga_ratings.parquet"))

display(manga_ratings)

,user,manga_id,manga_title,score,manga_idx,user_idx
0,wrath,14810,100 Dollar wa Yasu Sugiru,5,6001,29091
1,wrath,5259,99% Love,6,2831,29091
2,wrath,28621,Aoharu x Kikanjuu,6,9454,29091
3,wrath,51043,Ayakashi no Ou,8,11959,29091
4,wrath,10454,B.Ichi,7,4513,29091
...,...,...,...,...,...,...
1204811,Chamon7l,35733,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,7,10305,2942
1204812,Chamon7l,1023,Koukaku Kidoutai: The Ghost in the Shell,8,807,2942
1204813,Harv__,116778,Chainsaw Man,10,19889,6428
1204814,Harv__,123699,Death Note: Tokubetsu Yomikiri,9,20711,6428
